# 归一化方法

In [1]:
import torch
import torch.nn as nn

## 1.Batch Normalization
BatchNorm是沿着批次维度进行归一化的，它对每个特征在一个批次内进行归一化，使得每个特征的均值为0，方差为1。BatchNorm通常用于卷积神经网络中。

## 2.Layer Normalization
LayerNorm是沿着特征维度进行归一化的，它对每个样本的特征进行归一化，使得每个样本的特征均值为0，方差为1。LayerNorm通常用于RNN和Transformer中。

In [ ]:
class LayerNorm(nn.Module):
    """Layer Normalization 的手写实现。

    作用：对每个样本的最后一维做归一化，
    让特征分布更稳定，训练更容易收敛。
    """

    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()
        # normalized_shape 表示要归一化的特征维度大小，通常是最后一维的长度
        self.normalized_shape = normalized_shape
        # eps 是一个很小的数，用来防止除以 0，保证数值稳定
        self.eps = eps
        # gamma 是可学习的缩放参数，初始为 1，作用是恢复特征尺度
        self.gamma = nn.Parameter(torch.ones(normalized_shape))
        # beta 是可学习的平移参数，初始为 0，作用是恢复特征偏移
        self.beta = nn.Parameter(torch.zeros(normalized_shape))

    def forward(self, x):
        # 1. 计算均值：对最后一维求平均
        # keepdim=True 保持维度不变，方便后面做广播运算
        mean = x.mean(dim=-1, keepdim=True)

        # 2. 计算方差：衡量这一维上数值分布的离散程度
        # unbiased=False 表示使用总体方差，更符合深度学习中的常见实现
        var = x.var(dim=-1, keepdim=True, unbiased=False)

        # 3. 标准化：把数据变成均值为 0、方差为 1 的形式
        # 这样每个样本的特征分布会更稳定
        x_hat = (x - mean) / torch.sqrt(var + self.eps)

        # 4. 缩放和平移：让模型自己学习“归一化后应该是什么样子”
        # gamma 控制缩放，beta 控制偏移
        return self.gamma * x_hat + self.beta

## 3. RMS Normalization
RMSNorm是沿着特征维度进行归一化的，它对每个样本的特征进行归一化，使得每个样本的特征均值为0，方差为1。与LayerNorm不同的是，RMSNorm不使用均值，而是使用均方根（Root Mean Square）来进行归一化。这减少了同步开销，显著提升了前向和反向传播的计算速度。RMSNorm通常用于Transformer中。

具体实现
一定要注意：RMSNorm是对最后一维进行归一化，例如x.shape = [2, 5, 4096]，是对4096这一维度进行归一化。
1. nn.Parameter： 表示定义一个可学习参数 ， 初始化为全 1，表示一开始不改变归一化后的数值幅度。 
2. 为了防止溢出，在进行计算前转化为float()进行计算，在归一化之后再转换回去原来的格式
3. variance = x_float.pow(2).mean(dim=-1, keepdim=True)：先平方再取平均（mean）,在最后一个维度
4. 关于keepdim=True：如果不设置，在进行取平均值时会导致选择的那个dim消失了，而开启keepdim=True就可以保留维度：

    - 例如：x.shape = [2, 3, 4]
    - 不用 keepdim=True：variance.shape = [2, 3]
    - 使用 keepdim=True：variance.shape = [2, 3, 1]，这样在进行做差时：
    - x:        [2, 3, 4]
    - variance: [2, 3, 1]
    - 最后一维可以广播，所以计算成功

5.  广播：当两个 tensor 形状不完全一样，但满足一定规则时，PyTorch 会自动把较小的 tensor “扩展”成能参与计算的形状
6. 广播规则：从最后一维开始对齐。两个维度可以计算，当且仅当：两个维度相等，或者其中一个维度是 1，或者其中一个维度不存在

In [2]:
class RMSNorm(nn.Module):
    def __init__(self, hidden_size: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        # ==========================================
        # TODO 1: 定义可学习参数 weight，并初始化为全 1
        # 形状: [hidden_size]
        # 提示: 使用 nn.Parameter 包装张量使其可学习
        # ==========================================
        # self.weight = ???
        self.weight=nn.Parameter(torch.ones(hidden_size))

    def _norm(self, x: torch.Tensor) -> torch.Tensor:
        # ==========================================
        # TODO 2: 实现 RMSNorm 核心计算逻辑
        # 提示: 
        # 1. 为防止 FP16 溢出，需要在高精度下计算
        # 2. 计算输入的均方值（平方后求均值），注意保持维度以便广播
        # 3. 使用均方根的倒数进行归一化，torch.rsqrt 比 1/sqrt 更快
        # 4. 返回归一化后的结果（保持高精度，便于后续操作）
        # ==========================================
        # variance = ???
        x_fp32 = x if x.dtype == torch.float32 else x.float()
        variance=x_fp32.pow(2).mean(dim=-1,keepdim=True)
        return x_fp32 * torch.rsqrt(variance + self.eps)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # ==========================================
        # TODO 3: 先归一化，再缩放并转回输入精度
        # 提示: 调用 _norm 进行归一化后，乘以可学习的 weight，最后转回输入精度
        # ==========================================
        # weight = ???
        weight=self.weight.to(x.dtype)
        return (weight * self._norm(x)).to(x.dtype)


In [3]:
# 运行此单元格以测试你的实现
def test_rmsnorm():
    try:
        # 构造输入
        hidden_size = 512
        x = torch.randn(2, 16, hidden_size, dtype=torch.float16)  # FP16 输入模拟大模型
        
        # 测试你的实现
        my_norm = RMSNorm(hidden_size)
        # 将模型参数也转换为 FP16，对齐真实的工业半精度运行环境，防止发生隐式的 Type Promotion
        my_norm.to(x.dtype)
        my_out = my_norm(x)
        
        assert my_out.dtype == torch.float16, "输出类型必须与输入一致 (FP16)"
        assert my_out.shape == x.shape, "输出形状改变了！"
        
        # LLaMA 原版实现作为标准答案 (HuggingFace 提取)
        def hf_rmsnorm(hidden_states, weight, eps):
            input_dtype = hidden_states.dtype
            hidden_states = hidden_states.to(torch.float32)
            variance = hidden_states.pow(2).mean(-1, keepdim=True)
            hidden_states = hidden_states * torch.rsqrt(variance + eps)
            return weight.to(torch.float32) * hidden_states.to(input_dtype)
            
        hf_out = hf_rmsnorm(x, my_norm.weight, my_norm.eps)
        
        # 检查容差
        assert torch.allclose(my_out.float(), hf_out.float(), rtol=1e-3, atol=1e-4), "计算结果与 HuggingFace 不一致！"
        
        print("\n✅ All Tests Passed! RMSNorm 实现通过测试。")
        
    except NotImplementedError:
        print("请先完成 TODO 部分的代码！")
        raise
    except (AttributeError, NameError, TypeError) as e:
        if isinstance(e, AttributeError):
            print("代码未完成，无法找到 Parameter")
        elif isinstance(e, NameError):
            print("代码可能未完成，导致了变量未定义")
        else:
            print("代码可能未完成，导致了类型错误")
        raise NotImplementedError("请先完成 TODO 部分的代码！") from e
    except Exception as e:
        print(f"\n❌ 测试失败: {e}")

test_rmsnorm()


✅ All Tests Passed! RMSNorm 实现通过测试。
